# Task 4: Model Evaluation, Fine-Tuning Experiments & Test Prediction

1. Load best model from Task 3
2. Full evaluation (Dice, IoU, F1, Precision, Recall, Inference Speed)
3. Hyperparameter experiments
4. Comparison table
5. Test on provided test image

In [ ]:
import numpy as np, cv2, keras, tensorflow as tf, matplotlib.pyplot as plt, pandas as pd, time
from keras import layers, Model
from keras.applications import MobileNetV2
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import f1_score, precision_score, recall_score
print(f'TF: {tf.__version__}, GPU: {tf.config.list_physical_devices("GPU")}')
IMG_SIZE = 256; SEED = 42; np.random.seed(SEED)

In [ ]:
X_train_all = np.load('X_train_aug.npy')
y_train_all = np.load('y_train_aug.npy')
X_val = np.load('X_val.npy')
y_val = np.load('y_val.npy')
n_orig = len(X_train_all) // 2
X_train = X_train_all[:n_orig]
y_train = y_train_all[:n_orig]
print(f'Train: {X_train.shape}, Val: {X_val.shape}')

## 4.1 Loss Functions & Metrics

In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = keras.ops.cast(keras.ops.reshape(y_true, [-1]), 'float32')
    y_pred_f = keras.ops.cast(keras.ops.reshape(y_pred, [-1]), 'float32')
    inter = keras.ops.sum(y_true_f * y_pred_f)
    return (2.*inter+smooth)/(keras.ops.sum(y_true_f)+keras.ops.sum(y_pred_f)+smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    return keras.ops.mean(keras.losses.binary_crossentropy(y_true, y_pred)) + dice_loss(y_true, y_pred)

def focal_loss(y_true, y_pred, alpha=0.25, gamma=2.0):
    y_pred = keras.ops.clip(y_pred, 1e-7, 1.0 - 1e-7)
    bce = -(y_true * keras.ops.log(y_pred) + (1-y_true) * keras.ops.log(1-y_pred))
    p_t = y_true * y_pred + (1-y_true) * (1-y_pred)
    return keras.ops.mean(alpha * (1-p_t)**gamma * bce)

def focal_dice_loss(y_true, y_pred):
    return focal_loss(y_true, y_pred) + dice_loss(y_true, y_pred)

def iou_metric(y_true, y_pred, smooth=1e-6):
    y_true_f = keras.ops.cast(keras.ops.reshape(y_true, [-1]), 'float32')
    y_pred_f = keras.ops.cast(keras.ops.reshape(y_pred > 0.5, [-1]), 'float32')
    inter = keras.ops.sum(y_true_f * y_pred_f)
    union = keras.ops.sum(y_true_f) + keras.ops.sum(y_pred_f) - inter
    return (inter+smooth)/(union+smooth)

print('Loss functions and metrics defined.')

## 4.2 Data Generator & Model Builder

In [ ]:
class DataGen(keras.utils.Sequence):
    def __init__(self, images, masks, batch_size=8, augment=True, **kwargs):
        super().__init__(**kwargs)
        self.images, self.masks = images, masks
        self.batch_size, self.augment = batch_size, augment
        self.indices = np.arange(len(images))
        if augment: np.random.shuffle(self.indices)
    def __len__(self): return int(np.ceil(len(self.images)/self.batch_size))
    def __getitem__(self, idx):
        bi = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]
        imgs, msks = self.images[bi].copy(), self.masks[bi].copy()
        if self.augment:
            for i in range(len(imgs)): imgs[i], msks[i] = self._aug(imgs[i], msks[i])
        return imgs, msks
    def _aug(self, img, msk):
        if np.random.random()>0.5: img,msk = np.fliplr(img),np.fliplr(msk)
        if np.random.random()>0.5:
            a=np.random.uniform(-15,15); h,w=img.shape[:2]
            M=cv2.getRotationMatrix2D((w/2,h/2),a,1.0)
            img=cv2.warpAffine(img,M,(w,h),borderMode=cv2.BORDER_REFLECT)
            m2=msk[:,:,0]; m2=cv2.warpAffine(m2,M,(w,h),flags=cv2.INTER_NEAREST)
            msk=m2[:,:,np.newaxis]
        if np.random.random()>0.5: img=np.clip(img*np.random.uniform(0.7,1.3),0,1)
        if np.random.random()>0.7:
            k=np.random.choice([3,5])
            img=cv2.GaussianBlur(img,(k,k),0)
        return img.astype(np.float32), msk.astype(np.float32)
    def on_epoch_end(self):
        if self.augment: np.random.shuffle(self.indices)

print('DataGen ready.')

In [ ]:
def build_unet(input_shape=(256,256,3)):
    base = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    sn = ['block_1_expand_relu','block_3_expand_relu','block_6_expand_relu',
          'block_13_expand_relu','block_16_project']
    so = [base.get_layer(n).output for n in sn]
    enc = Model(inputs=base.input, outputs=so, name='encoder')
    enc.trainable = False
    inp = layers.Input(shape=input_shape)
    s1,s2,s3,s4,bn = enc(inp, training=False)
    def up(x, skip, f):
        x = layers.UpSampling2D(2)(x)
        x = layers.Concatenate()([x, skip])
        x = layers.SeparableConv2D(f,3,padding='same',activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.SpatialDropout2D(0.2)(x)
        x = layers.SeparableConv2D(f,3,padding='same',activation='relu')(x)
        x = layers.BatchNormalization()(x)
        return x
    x = up(bn,s4,256); x = up(x,s3,128); x = up(x,s2,64); x = up(x,s1,32)
    x = layers.UpSampling2D(2)(x)
    x = layers.Conv2D(16,3,padding='same',activation='relu')(x)
    x = layers.BatchNormalization()(x)
    out = layers.Conv2D(1,1,activation='sigmoid')(x)
    return Model(inp, out, name='UNet_MobileNetV2'), enc

print('build_unet defined.')

## 4.3 Evaluation Function

In [ ]:
def evaluate_model(model, X_v, y_v, name='Model'):
    preds_raw = model.predict(X_v, batch_size=8, verbose=0)
    preds_bin = (preds_raw > 0.5).astype(np.float32)
    yt = y_v.flatten().astype(int)
    yp = preds_bin.flatten().astype(int)
    smooth = 1e-6
    inter = np.sum(yt * yp)
    dice = (2*inter+smooth)/(np.sum(yt)+np.sum(yp)+smooth)
    union = np.sum(yt)+np.sum(yp)-inter
    iou = (inter+smooth)/(union+smooth)
    f1 = f1_score(yt, yp, zero_division=0)
    prec = precision_score(yt, yp, zero_division=0)
    rec = recall_score(yt, yp, zero_division=0)
    times = []
    for _ in range(25):
        t0 = time.time()
        _ = model.predict(X_v[:1], verbose=0)
        times.append((time.time()-t0)*1000)
    speed = np.mean(times[5:])
    r = {'Model':name,'Dice':dice,'IoU':iou,'F1':f1,'Precision':prec,'Recall':rec,'Speed_ms':speed}
    print(f'\n=== {name} ===')
    for k,v in r.items():
        if k!='Model': print(f'  {k}: {v:.4f}')
    return r, preds_raw, preds_bin

print('evaluate_model defined.')

## 4.4 Baseline Evaluation

In [ ]:
baseline_model, _ = build_unet()
baseline_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=bce_dice_loss,
                       metrics=[dice_coefficient, iou_metric, 'accuracy'])
baseline_model.load_weights('best_phase1.keras')
print('Loaded baseline model')
base_r, base_raw, base_bin = evaluate_model(baseline_model, X_val, y_val, 'Baseline (Task3)')
all_results = [base_r]

## 4.5 Baseline Predictions Visualization

In [ ]:
idx = np.random.choice(len(X_val), 6, replace=False)
fig, axes = plt.subplots(6, 4, figsize=(16, 24))
titles = ['Image','Ground Truth','Predicted (raw)','Predicted (binary)']
for i in range(6):
    axes[i,0].imshow(X_val[idx[i]])
    axes[i,1].imshow(y_val[idx[i],:,:,0], cmap='gray')
    axes[i,2].imshow(base_raw[idx[i],:,:,0], cmap='gray')
    axes[i,3].imshow(base_bin[idx[i],:,:,0], cmap='gray')
    for j in range(4):
        axes[i,j].axis('off')
        if i==0: axes[i,j].set_title(titles[j])
plt.suptitle('Baseline Predictions', fontsize=14)
plt.tight_layout()
plt.savefig('baseline_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.6 Hyperparameter Experiments

In [ ]:
def run_experiment(name, optimizer, loss_fn, batch_size=8, epochs=30, lr=1e-3):
    print(f'\n{"="*50} {name} {"="*50}')
    mdl, _ = build_unet()
    mdl.compile(optimizer=optimizer, loss=loss_fn, metrics=[dice_coefficient, iou_metric, 'accuracy'])
    tg = DataGen(X_train, y_train, batch_size, augment=True)
    vg = DataGen(X_val, y_val, batch_size, augment=False)
    cb = [
        ModelCheckpoint(f'exp_{name}.keras', monitor='val_dice_coefficient', mode='max',
                        save_best_only=True, verbose=0),
        EarlyStopping(monitor='val_dice_coefficient', mode='max', patience=10,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_dice_coefficient', mode='max', factor=0.5,
                          patience=5, min_lr=1e-7, verbose=1)
    ]
    t0 = time.time()
    hist = mdl.fit(tg, validation_data=vg, epochs=epochs, callbacks=cb, verbose=0)
    tt = time.time()-t0
    print(f'Time: {tt/60:.1f}min, Best val dice: {max(hist.history["val_dice_coefficient"]):.4f}')
    r, _, _ = evaluate_model(mdl, X_val, y_val, name)
    r['Train_min'] = round(tt/60,1)
    r['LR'] = lr
    r['Batch'] = batch_size
    return r

print('run_experiment defined.')

### Exp 1: SGD with Momentum

In [ ]:
r1 = run_experiment('SGD_momentum', keras.optimizers.SGD(learning_rate=1e-2, momentum=0.9),
                    bce_dice_loss, batch_size=8, lr=1e-2)
all_results.append(r1)

### Exp 2: Adam LR=1e-4

In [ ]:
r2 = run_experiment('Adam_lr1e-4', keras.optimizers.Adam(learning_rate=1e-4),
                    bce_dice_loss, batch_size=8, lr=1e-4)
all_results.append(r2)

### Exp 3: Batch Size 16

In [ ]:
r3 = run_experiment('Adam_batch16', keras.optimizers.Adam(learning_rate=1e-3),
                    bce_dice_loss, batch_size=16, lr=1e-3)
all_results.append(r3)

### Exp 4: Focal + Dice Loss

In [ ]:
r4 = run_experiment('Focal_Dice', keras.optimizers.Adam(learning_rate=1e-3),
                    focal_dice_loss, batch_size=8, lr=1e-3)
all_results.append(r4)

### Exp 5: Pure Dice Loss

In [ ]:
r5 = run_experiment('Pure_Dice', keras.optimizers.Adam(learning_rate=1e-3),
                    dice_loss, batch_size=8, lr=1e-3)
all_results.append(r5)

### Exp 6: AdamW with Weight Decay

In [ ]:
r6 = run_experiment('AdamW', keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
                    bce_dice_loss, batch_size=8, lr=1e-3)
all_results.append(r6)

## 4.7 Results Comparison

In [ ]:
df = pd.DataFrame(all_results)
cols = ['Model','Dice','IoU','F1','Precision','Recall','Speed_ms']
print('\n' + '='*80)
print('EXPERIMENT RESULTS')
print('='*80)
print(df[cols].to_string(index=False, float_format='%.4f'))
bi = df['Dice'].idxmax()
print(f'\nBest: {df.loc[bi,"Model"]} (Dice={df.loc[bi,"Dice"]:.4f}, IoU={df.loc[bi,"IoU"]:.4f})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
models = df['Model']; x = np.arange(len(models))
for ax, m, t, c in [(axes[0],'Dice',0.92,'steelblue'),(axes[1],'IoU',0.88,'coral'),(axes[2],'F1',0.90,'mediumseagreen')]:
    bars = ax.bar(x, df[m], color=c, edgecolor='black', alpha=0.8)
    ax.axhline(y=t, color='red', linestyle='--', alpha=0.7, label=f'Target {t}')
    ax.set_title(m); ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0,1); ax.legend(); ax.grid(axis='y',alpha=0.3)
    for bar,val in zip(bars,df[m]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.3f}', ha='center', fontsize=8)
plt.suptitle('Experiment Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('experiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.8 Observations

| Change | Expected Effect |
|--------|----------------|
| SGD vs Adam | SGD converges slower, may generalize better |
| Lower LR (1e-4) | More stable but slower convergence |
| Larger batch (16) | Faster training, may reduce generalization |
| Focal+Dice loss | Better handles class imbalance |
| Pure Dice loss | Directly optimizes target metric |
| AdamW | Weight decay regularizes, may reduce overfitting |

**Update the table above with your actual observations after running.**

## 4.9 Test on Provided Test Image

In [ ]:
test_raw = cv2.cvtColor(cv2.imread('Part 1Test Data - Prediction Image.jpeg'), cv2.COLOR_BGR2RGB)
print(f'Test image: {test_raw.shape}')
test_img = cv2.resize(test_raw, (256,256)).astype(np.float32)/255.0
test_input = np.expand_dims(test_img, 0)

# Use best model
best_name = df.loc[df['Dice'].idxmax(), 'Model']
print(f'Best model: {best_name}')
if best_name != 'Baseline (Task3)':
    best_mdl, _ = build_unet()
    best_mdl.compile(optimizer='adam', loss=bce_dice_loss, metrics=[dice_coefficient])
    best_mdl.load_weights(f'exp_{best_name}.keras')
else:
    best_mdl = baseline_model

t0 = time.time()
test_pred = best_mdl.predict(test_input, verbose=0)
print(f'Inference: {(time.time()-t0)*1000:.1f} ms')
test_mask = (test_pred[0,:,:,0] > 0.5).astype(np.uint8)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(test_raw); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(test_img); axes[1].set_title('Resized 256x256'); axes[1].axis('off')
axes[2].imshow(test_pred[0,:,:,0], cmap='hot'); axes[2].set_title('Heatmap'); axes[2].axis('off')
overlay = test_img.copy()
overlay[test_mask==1] = [0,1,0]
blended = cv2.addWeighted(test_img, 0.6, overlay, 0.4, 0)
axes[3].imshow(blended); axes[3].set_title('Mask Overlay'); axes[3].axis('off')
plt.suptitle(f'Test Prediction ({best_name})', fontsize=14)
plt.tight_layout()
plt.savefig('test_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.10 Performance Report

In [ ]:
print('='*60)
print('PERFORMANCE REPORT')
print('='*60)
print(f'Dataset: 409 images, 256x256, bounding-box annotations')
print(f'Train/Val: {len(X_train)}/{len(X_val)}')
br = df.loc[df['Dice'].idxmax()]
print(f'\nBest Model: {br["Model"]}')
print(f'  Dice:      {br["Dice"]:.4f} (target >0.92)')
print(f'  IoU:       {br["IoU"]:.4f} (target >0.88)')
print(f'  F1:        {br["F1"]:.4f} (target >0.90)')
print(f'  Speed:     {br["Speed_ms"]:.1f} ms (target <100ms)')
print(f'\nAll Results:')
print(df[cols].to_string(index=False, float_format='%.4f'))
print('\nTask 4 Complete! Ready for Task 5 (Streamlit App).')